# Text Generation with Markov Chains

This notebook makes the project documentation reproducible, then works through a text-derived Markov chain using explicit matrices. The convention throughout is row-stochastic: `P[i, j]` is the probability of moving from word `i` to word `j`, and row distributions evolve as $x_{k+1} = x_k P$. The examples use the `wrap` dead-end policy unless stated otherwise.

## 1. Create a README File

The README is also kept in the repository so the project can be understood without opening this notebook. This cell writes the same essential project information using Python file I/O. It is safe to rerun: the content is deterministic and replaces the same file.

In [6]:
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "markov").is_dir():
    ROOT = Path.cwd().parent

readme_text = """# Text Generation with Markov Chains

This is my Linear Algebra project about generating text with Markov chains. The program counts which words follow each other, turns those counts into a transition matrix, and uses the matrix to generate text and study long-run word frequencies. I use NumPy, Matplotlib, and SymPy, but no machine-learning libraries.

## How the model works

Each unique lowercase word is a state. In the transition matrix, `P[i, j]` is the probability that word `j` follows word `i`. The matrix is row-stochastic, so a row distribution changes by `x_(k+1) = x_k P`.

If the last word has no observed next word, the default is to wrap back to the first word. Other dead-end choices are available in the code and can change the results.

## Run the project

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m examples.small_example
python -m examples.full_example
```

Run the tests with `python -m pytest -q`. The notebook walks through the calculations, and the full example saves its plots in `figures/`.

## Files

- `markov/` contains the code for the model and calculations.
- `examples/` contains a small worked example and a longer example.
- `data/sample.txt` is the plain-text input story for the full example.
- `tests/` contains checks for the matrix, steady state, and generated text.
- `notebook.ipynb` contains the project walkthrough.
"""

readme_path = ROOT / "README.md"
readme_path.write_text(readme_text, encoding="utf-8")
print(f"Wrote {readme_path}")

Wrote /Users/adenli/markov-text-generation/README.md


## 2. Add README Creation to the Project Plan

The project plan lives in `PROJECT_PLAN.md`. This cell checks that README creation is represented there and writes the completed status explicitly, so the plan and project files stay in sync.

In [7]:
plan_path = ROOT / "PROJECT_PLAN.md"
plan_text = plan_path.read_text(encoding="utf-8")
readme_plan_line = "7. **README creation**: document setup, conventions, commands, project layout, and interpretation of results. **Complete**"
if "7. **README creation**" not in plan_text:
    plan_text = plan_text.replace("## Plan Review", f"{readme_plan_line}\n\n## Plan Review")
elif readme_plan_line not in plan_text:
    lines = plan_text.splitlines()
    plan_text = "\n".join(
        readme_plan_line if line.startswith("7. **README creation**") else line
        for line in lines
    ) + "\n"
plan_path.write_text(plan_text, encoding="utf-8")
print(f"README creation is recorded in {plan_path}")

README creation is recorded in /Users/adenli/markov-text-generation/PROJECT_PLAN.md


## 3. Small Example: Exact Transition Probabilities

The tokenizer lowercases words and removes punctuation. The state order is first appearance in the text. A state that has no outgoing bigrams uses the documented `wrap` default: it transitions back to the first token. This example has seven states; its transition graph has period 1, so it is aperiodic, even though it has no literal self-loop.

In [3]:
import numpy as np
import sympy as sp

from markov import (
    diagnose_chain,
    distribution_history,
    empirical_distribution,
    exact_transition_matrix,
    format_matrix,
    format_stationary_system,
    generate,
    matrix_powers,
    plot_convergence,
    plot_frequency_comparison,
    plot_transition_heatmap,
    steady_state_by_eigenvector,
    steady_state_by_system,
    text_to_matrix,
    tokenize,
)

small_text = "the cat sat next to the already sat dog"
small_counts, small_transition, small_words = text_to_matrix(small_text)
small_exact = exact_transition_matrix(small_counts)

print("State order:", small_words)
print("\nBigram counts:")
print(format_matrix(small_counts, small_words, precision=0))
print("\nExact row-stochastic transition matrix:")
print(format_matrix(small_exact, small_words))
print("\nRow sums:", small_transition.sum(axis=1))
assert np.allclose(small_transition.sum(axis=1), 1.0)

State order: ['the', 'cat', 'sat', 'next', 'to', 'already', 'dog']

Bigram counts:
from / to  the  cat  sat  next  to  already  dog
the          0    1    0     0   0        1    0
cat          0    0    1     0   0        0    0
sat          0    0    0     1   0        0    1
next         0    0    0     0   1        0    0
to           1    0    0     0   0        0    0
already      0    0    1     0   0        0    0
dog          1    0    0     0   0        0    0

Exact row-stochastic transition matrix:
from / to  the  cat  sat  next  to  already  dog
the          0  1/2    0     0   0      1/2    0
cat          0    0    1     0   0        0    0
sat          0    0    0   1/2   0        0  1/2
next         0    0    0     0   1        0    0
to           1    0    0     0   0        0    0
already      0    0    1     0   0        0    0
dog          1    0    0     0   0        0    0

Row sums: [1. 1. 1. 1. 1. 1. 1.]


## 4. Stationary State as a Linear System

For a row-stochastic matrix, a stationary column vector satisfies $(P^T-I)\pi^T=0$ together with $\sum_i\pi_i=1$. The last equation replaces one redundant balance equation, giving a solvable augmented system. SymPy keeps the small example exact.

In [5]:
print(format_stationary_system(small_exact, small_words))
small_by_system = steady_state_by_system(small_exact)
small_by_eigenvector = steady_state_by_eigenvector(small_exact)
print("\nSystem solution:", small_by_system.T)
print("Eigenvector solution:", small_by_eigenvector.T)
assert small_by_system == small_by_eigenvector
assert small_by_system == sp.Matrix([
    sp.Rational(2, 9),
    sp.Rational(1, 9),
    sp.Rational(2, 9),
    sp.Rational(1, 9),
    sp.Rational(1, 9),
    sp.Rational(1, 9),
    sp.Rational(1, 9),
])

A = [P^T - I; 1^T], b = [0; 1]
⎡-1   0    0   0   1   0   1   0⎤
⎢                               ⎥
⎢1/2  -1   0   0   0   0   0   0⎥
⎢                               ⎥
⎢ 0   1   -1   0   0   1   0   0⎥
⎢                               ⎥
⎢ 0   0   1/2  -1  0   0   0   0⎥
⎢                               ⎥
⎢ 0   0    0   1   -1  0   0   0⎥
⎢                               ⎥
⎢1/2  0    0   0   0   -1  0   0⎥
⎢                               ⎥
⎢ 0   0   1/2  0   0   0   -1  0⎥
⎢                               ⎥
⎣ 1   1    1   1   1   1   1   1⎦
Variables: pi_the, pi_cat, pi_sat, pi_next, pi_to, pi_already, pi_dog

System solution: Matrix([[2/9, 1/9, 2/9, 1/9, 1/9, 1/9, 1/9]])
Eigenvector solution: Matrix([[2/9, 1/9, 2/9, 1/9, 1/9, 1/9, 1/9]])


## 5. Matrix Powers and Distribution Evolution

Powers $P^k$ describe how the next-state distribution changes after several transitions. Every row of $P^k$ is the distribution after $k$ steps from the corresponding one-hot starting state. We also follow one explicit initial distribution, $x_0$, using $x_k=x_0P^k$.

In [8]:
small_powers = matrix_powers(small_transition)
for exponent, power in small_powers.items():
    print(f"\nP^{exponent}:")
    print(format_matrix(power, small_words, precision=3))

small_initial = np.zeros(len(small_words))
small_initial[small_words.index("the")] = 1.0
small_history = distribution_history(small_initial, small_transition, 10)
print("\nx_0:", small_history[0])
print("x_10:", small_history[-1])
print("pi:", np.asarray(small_by_system, dtype=float).reshape(-1))


P^1:
from / to    the    cat    sat   next     to  already    dog
the        0.000  0.500  0.000  0.000  0.000    0.500  0.000
cat        0.000  0.000  1.000  0.000  0.000    0.000  0.000
sat        0.000  0.000  0.000  0.500  0.000    0.000  0.500
next       0.000  0.000  0.000  0.000  1.000    0.000  0.000
to         1.000  0.000  0.000  0.000  0.000    0.000  0.000
already    0.000  0.000  1.000  0.000  0.000    0.000  0.000
dog        1.000  0.000  0.000  0.000  0.000    0.000  0.000

P^2:
from / to    the    cat    sat   next     to  already    dog
the        0.000  0.000  1.000  0.000  0.000    0.000  0.000
cat        0.000  0.000  0.000  0.500  0.000    0.000  0.500
sat        0.500  0.000  0.000  0.000  0.500    0.000  0.000
next       1.000  0.000  0.000  0.000  0.000    0.000  0.000
to         0.000  0.500  0.000  0.000  0.000    0.500  0.000
already    0.000  0.000  0.000  0.500  0.000    0.000  0.500
dog        0.000  0.500  0.000  0.000  0.000    0.500  0.000

P^5:
from /

## 6. Reducibility and Periodicity Diagnostics

A reducible chain has multiple communicating components and can have more than one stationary distribution. A periodic recurrent class can have a stationary distribution even while its powers oscillate. The two-state chain below alternates deterministically, so $P^k$ does not converge.

In [9]:
periodic_transition = np.array([[0.0, 1.0], [1.0, 0.0]])
periodic_report = diagnose_chain(periodic_transition, ["left", "right"])
reducible_report = diagnose_chain(np.eye(2), ["fixed A", "fixed B"])
print("Periodic example:", periodic_report)
print("Reducible example:", reducible_report)
assert periodic_report["periodic"]
assert reducible_report["reducible"]

Periodic example: {'reducible': False, 'periodic': True, 'components': [['right', 'left']], 'closed_classes': [['right', 'left']], 'recurrent_periods': {('right', 'left'): 2}, 'warnings': ['A recurrent class is periodic; P^k may not converge.']}
Reducible example: {'reducible': True, 'periodic': False, 'components': [['fixed A'], ['fixed B']], 'closed_classes': [['fixed A'], ['fixed B']], 'recurrent_periods': {('fixed A',): 1, ('fixed B',): 1}, 'warnings': ['Chain is reducible; stationary distributions may not be unique.', 'Multiple closed classes imply multiple stationary distributions.']}


## 7. Long Text, Simulation, and Frequency Comparison

The text in `data/sample.txt` is a finite sample that defines the bigram transitions; it is not expected to have the same frequency distribution as the chain's long-run behavior. A seeded long simulation provides an empirical estimate of the stationary distribution. Replace the contents of that file with your own plain-text story to analyze a different input.

In [ ]:
sample_text = (ROOT / "data" / "sample.txt").read_text(encoding="utf-8")
counts, transition, words = text_to_matrix(sample_text, dead_end_policy="wrap")
exact_transition = exact_transition_matrix(counts)
report = diagnose_chain(transition, words)
print(f"States: {len(words)}; P shape: {transition.shape}")
print("Maximum row-sum error:", np.max(np.abs(transition.sum(axis=1) - 1.0)))
print(
    "Diagnostics:",
    {
        "reducible": report["reducible"],
        "periodic": report["periodic"],
        "component_count": len(report["components"]),
        "closed_class_count": len(report["closed_classes"]),
        "recurrent_periods": list(report["recurrent_periods"].values()),
    },
)
for warning in report["warnings"]:
    print("WARNING:", warning)

source_tokens = tokenize(sample_text)
initial = np.zeros(len(words))
initial[words.index(source_tokens[0])] = 1.0
history = distribution_history(initial, transition, 50)
source_frequencies = empirical_distribution(source_tokens, words)
generated_tokens = generate(transition, words, source_tokens[0], 100_000, seed=2026)
generated_frequencies = empirical_distribution(generated_tokens, words)

try:
    exact_stationary = steady_state_by_system(exact_transition)
    eigen_stationary = steady_state_by_eigenvector(exact_transition)
    assert exact_stationary == eigen_stationary
    stationary = np.asarray(exact_stationary, dtype=float).reshape(-1)
except ValueError as error:
    stationary = None
    print("No unique stationary vector:", error)

if stationary is not None:
    print("\nWord frequencies (stationary / source text / 100,000 generated states):")
    for index, word in enumerate(words):
        print(
            f"{word:>12}: {stationary[index]:.4f} / "
            f"{source_frequencies[index]:.4f} / {generated_frequencies[index]:.4f}"
        )
    print("\nPowers and distance from each row to pi:")
    for exponent, power in matrix_powers(transition).items():
        distances = np.linalg.norm(power - stationary, axis=1)
        print(
            f"k={exponent:>2}: start row distance={distances[words.index(source_tokens[0])]:.5f}; "
            f"maximum row distance={distances.max():.5f}"
        )

    figures = ROOT / "figures"
    plot_transition_heatmap(transition, words, figures / "transition_heatmap.png")
    plot_frequency_comparison(
        words, stationary, source_frequencies, figures / "frequency_comparison.png"
    )
    plot_convergence(history, stationary, figures / "convergence.png")
    print("\nSaved plots:", sorted(path.name for path in figures.glob("*.png")))
    print(
        "Generated frequencies are close to the stationary distribution because this "
        "sample chain has one aperiodic closed class. Source-text frequencies differ "
        "because they count one finite paragraph rather than long-run Markov visits."
    )